In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import pandas as pd
import numpy as np

file_path = "/content/drive/MyDrive/Sophomore Year/IML_Fall2025_SkillVersusLuck/Table Generation/serie_a.csv"
df = pd.read_csv(file_path)
df.sample(n=20)

,season,date,home_team,away_team,hometeamgoals,awayteamgoals,hometeamresult,home_team_points,away_team_points,OddHome,OddDraw,OddAway
1976,2019,2020-07-08,Roma,Parma,2.0,1.0,1,3.0,0.0,1.51,4.38,6.29
7630,2004,2005-05-08,Atalanta,Messina,2.0,1.0,1,3.0,0.0,1.57,3.50,5.00
7111,2006,2006-11-12,Catania,Livorno,3.0,2.0,1,3.0,0.0,2.40,2.80,3.30
5530,2010,2011-01-06,Palermo,Sampdoria,3.0,0.0,1,3.0,0.0,1.91,3.30,4.20
5049,2011,2012-03-17,Parma,Milan,0.0,2.0,-1,0.0,3.0,5.50,3.50,1.67
2811,2017,2018-02-04,Atalanta,Chievo,1.0,0.0,1,3.0,0.0,1.36,4.75,9.50
4794,2012,2012-12-02,Lazio,Parma,2.0,1.0,1,3.0,0.0,1.73,3.60,4.75
6379,2008,2008-10-29,Torino,Atalanta,2.0,1.0,1,3.0,0.0,2.40,3.00,3.20
8199,2002,2003-03-02,Roma,Empoli,3.0,1.0,1,3.0,0.0,1.44,3.60,6.50
4700,2012,2013-02-10,Atalanta,Catania,0.0,0.0,0,1.0,1.0,2.20,3.20,3.40


In [20]:
# --- Ensure types ---

df["season"] = pd.to_numeric(df["season"], errors="coerce").astype("Int64")
for c in ["hometeamgoals","awayteamgoals"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

use = df[df["hometeamgoals"].notna() & df["awayteamgoals"].notna()].copy()


In [21]:
# --- Compare goals scored to compute results ---

hg = use["hometeamgoals"].astype(float)
ag = use["awayteamgoals"].astype(float)

home_win  = (hg >  ag)
home_draw = (hg == ag)
home_loss = (hg <  ag)


In [22]:
# --- Building per-team rows (home & away) for ALL seasons ---

home = pd.DataFrame({
    "season": use["season"].astype(int),
    "team":   use["home_team"],
    "points": (home_win*3 + home_draw*1).astype(int),
    "wins":   home_win.astype(int),
    "draws":  home_draw.astype(int),
    "losses": home_loss.astype(int),
    "gf": use["hometeamgoals"].astype(int),
    "ga": use["awayteamgoals"].astype(int),
})

away = pd.DataFrame({
    "season": use["season"].astype(int),
    "team":   use["away_team"],
    "points": (home_loss*3 + home_draw*1).astype(int),  # away gets 3 when home loses
    "wins":   home_loss.astype(int),
    "draws":  home_draw.astype(int),
    "losses": home_win.astype(int),
    "gf": use["awayteamgoals"].astype(int),
    "ga": use["hometeamgoals"].astype(int),
})

long = pd.concat([home, away], ignore_index=True)


In [23]:
# --- Combined to end-of-season table for each (season, team) ---

agg = (long.groupby(["season","team"], as_index=False).sum(numeric_only=True))
agg["gd"] = agg["gf"] - agg["ga"]


In [24]:
# --- Rank within each season: Points ↓, GD ↓, GF ↓, Team ↑ ---

agg = agg.sort_values(
    ["season","points","gd","gf","team"],
    ascending=[True, False, False, False, True],
    kind="mergesort"
)
agg["rank"] = agg.groupby("season").cumcount() + 1


In [25]:
# --- Final view (only required columns) ---

standings_all = (agg.rename(columns={"team":"team_name"})
                   [["season","team_name","points","wins","draws","losses","gf","ga","gd","rank"]]
                   .sort_values(["season","rank"])
                   .reset_index(drop=True))


In [26]:
display(standings_all[standings_all["season"] == 2019].sort_values("rank").head(18))


,season,team_name,points,wins,draws,losses,gf,ga,gd,rank
372,2019,Juventus,83,26,5,7,76,43,33,1
373,2019,Inter,82,24,10,4,81,36,45,2
374,2019,Atalanta,78,23,9,6,98,48,50,3
375,2019,Lazio,78,24,6,8,79,42,37,4
376,2019,Roma,70,21,7,10,77,51,26,5
377,2019,Milan,66,19,9,10,63,46,17,6
378,2019,Napoli,62,18,8,12,61,50,11,7
379,2019,Sassuolo,51,14,9,15,69,63,6,8
380,2019,Fiorentina,49,12,13,13,51,48,3,9
381,2019,Parma,49,14,7,17,56,57,-1,10


In [27]:
out_path = "/content/drive/MyDrive/Sophomore Year/IML_Fall2025_SkillVersusLuck/Table Generation/serie_a_standings_all_seasons.csv"
standings_all.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: /content/drive/MyDrive/Sophomore Year/IML_Fall2025_SkillVersusLuck/Table Generation/serie_a_standings_all_seasons.csv
